# WMH Brain Lesion — Boundary Loss Ablation

비교: `ce_dice` / `plwce_dice` (baseline) vs Boundary Loss 변형 8종

In [ ]:
# === Cell 0: 환경 설정 ===
import subprocess, sys
for pkg in ['segmentation-models-pytorch','nibabel','openpyxl','kagglehub','scipy','optuna']:
    subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])
import os,warnings,json,random,glob
warnings.filterwarnings('ignore')
os.environ['TQDM_DISABLE']='1'
import numpy as np, nibabel as nib, cv2
from tqdm import tqdm
import torch,torch.nn as nn,torch.optim as optim
from torch.utils.data import Dataset,DataLoader
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import segmentation_models_pytorch as smp
sys.path.insert(0,'/root/imbalanced-data-LWCE/medical_data')
from custom_losses import get_loss_function
DOMAIN='wmh'; NUM_CLASSES=2; CLASS_NAMES=['Background','WMH']
IMG_SIZE=256; BATCH_SIZE=16; NUM_WORKERS=0; SEED=42; BG_ONLY_RATIO=0.3
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
RESULTS_DIR='/root/imbalanced-data-LWCE/medical_data/boundary_ablation/results/wmh'
os.makedirs(RESULTS_DIR,exist_ok=True)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}'); print('환경 설정 완료')

In [ ]:
# === Cell 1: 데이터 로드 ===
import kagglehub
_kaggle_path=kagglehub.dataset_download("farahmo/wmh-dataset")
print("Path:",_kaggle_path)
RAW_DIR=_kaggle_path
flair_check=glob.glob(os.path.join(RAW_DIR,'**','FLAIR.nii*'),recursive=True)
print(f'FLAIR 파일 {len(flair_check)}개 발견')

def find_wmh_cases(base_dir):
    cases=[]
    for flair_path in glob.glob(os.path.join(base_dir,'**','FLAIR.nii*'),recursive=True):
        case_dir=os.path.dirname(flair_path); parent=os.path.dirname(case_dir)
        t1_path=os.path.join(case_dir,'T1.nii.gz')
        if not os.path.exists(t1_path): t1_path=os.path.join(case_dir,'T1.nii')
        wmh_path=next((p for p in [
            os.path.join(parent,'wmh.nii.gz'),os.path.join(parent,'wmh.nii'),
            os.path.join(case_dir,'wmh.nii.gz'),os.path.join(case_dir,'wmh.nii'),
            os.path.join(parent,'lesion.nii.gz'),os.path.join(parent,'mask.nii.gz'),
        ] if os.path.exists(p)),None)
        if os.path.exists(t1_path) and wmh_path: cases.append((flair_path,t1_path,wmh_path))
    return cases

cases=find_wmh_cases(RAW_DIR); print(f'케이스: {len(cases)}')

SLICE_DIR='/tmp/wmh_slices'; os.makedirs(SLICE_DIR,exist_ok=True)
def percentile_normalize(arr,p_low=1,p_high=99):
    fg=arr[arr>arr.mean()*0.1]
    lo=np.percentile(fg,p_low) if len(fg)>0 else arr.min()
    hi=np.percentile(fg,p_high) if len(fg)>0 else arr.max()
    return np.clip((np.clip(arr,lo,hi)-lo)/(hi-lo+1e-8),0,1).astype(np.float32)

existing=glob.glob(os.path.join(SLICE_DIR,'*.npz'))
if len(existing)<100 and len(cases)>0:
    print('슬라이스 전처리 중...')
    for ci,(fp,tp,wp) in enumerate(tqdm(cases)):
        fv=percentile_normalize(nib.load(fp).get_fdata().astype(np.float32))
        tv=percentile_normalize(nib.load(tp).get_fdata().astype(np.float32))
        wv=(nib.load(wp).get_fdata()>0.5).astype(np.int64)
        for s in range(fv.shape[2]):
            fs=cv2.resize(fv[:,:,s],(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_LINEAR)
            ts=cv2.resize(tv[:,:,s],(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_LINEAR)
            ws=cv2.resize(wv[:,:,s].astype(np.uint8),(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST).astype(np.int64)
            np.savez_compressed(os.path.join(SLICE_DIR,f'case{ci:03d}_s{s:04d}.npz'),
                                image=np.stack([fs,ts,fs],axis=0).astype(np.float32),label=ws)
    print('완료')
elif len(existing)>=100: print(f'캐시 사용: {len(existing)}개')

all_slices=sorted(glob.glob(os.path.join(SLICE_DIR,'*.npz')))
wmh_slices=[f for f in all_slices if np.load(f)['label'].sum()>0]
bg_slices=[f for f in all_slices if np.load(f)['label'].sum()==0]
n_bg=min(len(bg_slices),int(len(wmh_slices)*BG_ONLY_RATIO))
slice_files=sorted(wmh_slices+random.sample(bg_slices,n_bg))
print(f'WMH:{len(wmh_slices)} BG:{n_bg} 총:{len(slice_files)}')

case_ids=sorted(set(int(os.path.basename(f).split('_')[0][4:]) for f in slice_files))
tr_ids,tmp=train_test_split(case_ids,test_size=0.2,random_state=SEED)
val_ids,test_ids=train_test_split(tmp,test_size=0.5,random_state=SEED)
tr_set,val_set,test_set=set(tr_ids),set(val_ids),set(test_ids)
def gc(f): return int(os.path.basename(f).split('_')[0][4:])
tr_files=[f for f in slice_files if gc(f) in tr_set]
val_files=[f for f in slice_files if gc(f) in val_set]
test_files=[f for f in slice_files if gc(f) in test_set]
print(f'Train:{len(tr_files)} Val:{len(val_files)} Test:{len(test_files)}')

class WMHDataset(Dataset):
    def __init__(self,files,augment=False): self.files=files; self.augment=augment
    def __len__(self): return len(self.files)
    def __getitem__(self,idx):
        d=np.load(self.files[idx]); img,lbl=d['image'].astype(np.float32),d['label'].astype(np.int64)
        if self.augment:
            if random.random()>0.5: img=np.flip(img,axis=2).copy(); lbl=np.fliplr(lbl).copy()
            if random.random()>0.5: img=np.flip(img,axis=1).copy(); lbl=np.flipud(lbl).copy()
        return torch.from_numpy(img),torch.from_numpy(lbl)

train_loader=DataLoader(WMHDataset(tr_files,augment=True),batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=True)
val_loader=DataLoader(WMHDataset(val_files),batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True)
test_loader=DataLoader(WMHDataset(test_files),batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True)
print('DataLoader 완료')

In [ ]:
# === Cell 2: 클래스 비율 계산 ===
print('클래스 비율 계산 중...')
class_counts=np.zeros(NUM_CLASSES,dtype=np.int64)
for fp in tqdm(tr_files):
    l=np.load(fp)['label'].astype(np.int64)
    class_counts[0]+=int((l==0).sum()); class_counts[1]+=int((l==1).sum())
class_counts=class_counts.tolist()
total=sum(class_counts)
for c,(n,cnt) in enumerate(zip(CLASS_NAMES,class_counts)):
    print(f'  [{c}] {n:<12}: {cnt:>15,} ({100*cnt/total:.4f}%)')
print(f'BG:WMH={class_counts[0]/class_counts[1]:.1f}:1')
print(f'class_counts={class_counts}')

In [ ]:
# === Cell 3: 모델 정의 ===

def to_2ch_logits(p):
    return torch.cat([-p, p], dim=1)

def build_model():
    return smp.Unet(
        encoder_name='resnet34', encoder_weights='imagenet',
        in_channels=3, classes=1, activation=None,
    ).to(device)

def compute_val_dice(model, loader):
    model.eval()
    tp = fp = fn = 0
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            prob = torch.sigmoid(model(imgs)[:, 0])
            pred = (prob > 0.5).long()
            tp += ((pred==1)&(masks==1)).sum().item()
            fp += ((pred==1)&(masks==0)).sum().item()
            fn += ((pred==0)&(masks==1)).sum().item()
    return float(2*tp/(2*tp+fp+fn+1e-8))

def compute_val_metrics(model, loader):
    model.eval()
    all_probs, all_preds, all_labels = [], [], []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(device)
            prob = torch.sigmoid(model(imgs)[:, 0]).cpu().numpy()
            pred = (prob > 0.5).astype(np.int64)
            all_probs.append(prob.flatten())
            all_preds.append(pred.flatten())
            all_labels.append(masks.numpy().flatten())
    all_probs  = np.concatenate(all_probs)
    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    TP = ((all_preds==1)&(all_labels==1)).sum()
    FP = ((all_preds==1)&(all_labels==0)).sum()
    TN = ((all_preds==0)&(all_labels==0)).sum()
    FN = ((all_preds==0)&(all_labels==1)).sum()
    dice = 2*TP/(2*TP+FP+FN+1e-8)
    sens = TP/(TP+FN+1e-8)
    spec = TN/(TN+FP+1e-8)
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except Exception:
        auc = 0.0
    return {'Dice':float(dice),'Sensitivity':float(sens),'Specificity':float(spec),'AUC':float(auc)}

print('모델 + 유틸리티 함수 준비 완료')

In [ ]:
# === Cell 4: 학습 함수 ===

def train_model(loss_name, alpha=1.0, gamma=2.0, epochs=30, lr=1e-4,
                subset_ratio=1.0, tag=''):
    model     = build_model()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = get_loss_function(loss_name, class_counts=class_counts, alpha=alpha, gamma=gamma)
    name = f'{loss_name}_a{alpha:.2f}' if alpha != 1.0 else loss_name
    if tag: name = f'{tag}_{name}'
    print(f"\n{'='*60}\n{name}  (epochs={epochs})\n{'='*60}")
    if subset_ratio < 1.0:
        n = max(1, int(len(train_loader.dataset) * subset_ratio))
        sub_ds = torch.utils.data.Subset(
            train_loader.dataset, random.sample(range(len(train_loader.dataset)), n))
        loader = DataLoader(sub_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    else:
        loader = train_loader
    history = {'loss':[], 'val_dice':[]}
    best_dice = 0.0
    save_path = f'/tmp/ba_wmh_{name}.pth'
    for epoch in range(epochs):
        # --- Boundary Loss annealing: BL 비중 0 → 0.5 선형 증가 ---
        if criterion.boundary_loss is not None:
            alpha_t = min(epoch / epochs, 0.5)
            criterion.set_boundary_alpha(alpha_t)
        model.train()
        epoch_loss = 0.0
        for imgs, masks in tqdm(loader, desc=f'Ep{epoch+1:02d}/{epochs}', leave=False):
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            loss = criterion(to_2ch_logits(model(imgs)), masks)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()
        avg_loss = epoch_loss / len(loader)
        val_dice = compute_val_dice(model, val_loader)
        history['loss'].append(avg_loss)
        history['val_dice'].append(val_dice)
        print(f'Ep{epoch+1:02d} | Loss: {avg_loss:.4f} | Val Dice: {val_dice:.4f}', end='')
        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.state_dict(), save_path)
            print('  <- Best!', end='')
        print()
    model.load_state_dict(torch.load(save_path, weights_only=True))
    print(f'최고 Val Dice: {best_dice:.4f}')
    return model, history, best_dice

print('train_model() 준비 완료')

In [ ]:
# === Cell 5: Optuna — PLWCE alpha 탐색 (Boundary Loss 환경) ===
# Dice+BL 조합에서는 alpha 최적값이 기존 Dice 전용 실험과 다를 수 있음.
# 두 대표 손실 함수에 대해 별도 탐색:
#   plwce_dice_boundary  → alpha_with_dice    (plwce_dice_log_boundary에도 재사용)
#   plwce_boundary       → alpha_without_dice (plwce_log_boundary에도 재사용)
import optuna, traceback
optuna.logging.set_verbosity(optuna.logging.WARNING)
os.environ['TQDM_DISABLE'] = '1'

ALPHA_LOW    = 1.0
ALPHA_HIGH   = 20.0
PROXY_EPOCHS = 5
PROXY_RATIO  = 0.15
N_TRIALS     = 20

def make_objective(loss_name):
    def objective(trial):
        alpha = trial.suggest_float('alpha', ALPHA_LOW, ALPHA_HIGH)
        try:
            _, _, dice = train_model(loss_name, alpha=alpha,
                                     epochs=PROXY_EPOCHS, subset_ratio=PROXY_RATIO)
            return dice
        except Exception:
            traceback.print_exc()
            return None
    return objective

# --- plwce_dice_boundary ---
print('Optuna: plwce_dice_boundary ...')
study_pdb = optuna.create_study(direction='maximize')
study_pdb.optimize(make_objective('plwce_dice_boundary'), n_trials=N_TRIALS)
best_trials_pdb = [t for t in study_pdb.trials if t.value is not None]
best_alpha_with_dice = (
    best_trials_pdb[int(np.argmax([t.value for t in best_trials_pdb]))].params['alpha']
    if best_trials_pdb else 3.275
)
print(f'  best alpha (with Dice): {best_alpha_with_dice:.3f}')

# --- plwce_boundary (no Dice) ---
print('Optuna: plwce_boundary (no Dice) ...')
study_pb = optuna.create_study(direction='maximize')
study_pb.optimize(make_objective('plwce_boundary'), n_trials=N_TRIALS)
best_trials_pb = [t for t in study_pb.trials if t.value is not None]
best_alpha_without_dice = (
    best_trials_pb[int(np.argmax([t.value for t in best_trials_pb]))].params['alpha']
    if best_trials_pb else 3.275
)
print(f'  best alpha (no Dice):   {best_alpha_without_dice:.3f}')

# JSON 저장
optuna_data = {
    'plwce_dice_boundary': {'best_alpha': best_alpha_with_dice},
    'plwce_boundary':      {'best_alpha': best_alpha_without_dice},
}
with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_boundary_optuna.json'), 'w') as f:
    json.dump(optuna_data, f, indent=2)
print('Optuna 결과 저장 완료')

In [ ]:
# === Cell 6: Boundary Ablation 전체 학습 ===

FINAL_EPOCHS = 50
FINAL_LR     = 1e-4

# --- Optuna 결과 로드 (Cell 5 미실행 시 JSON fallback) ---
try:
    _ = best_alpha_with_dice
except NameError:
    try:
        with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_boundary_optuna.json')) as f:
            d = json.load(f)
        best_alpha_with_dice    = d['plwce_dice_boundary']['best_alpha']
        best_alpha_without_dice = d['plwce_boundary']['best_alpha']
        print(f'Optuna 로드: with_dice={best_alpha_with_dice:.3f}, no_dice={best_alpha_without_dice:.3f}')
    except FileNotFoundError:
        best_alpha_with_dice    = 3.275
        best_alpha_without_dice = 3.275
        print(f'Optuna 미실행 → fallback alpha=3.275')

experiments = [
    # (loss_name,                  alpha,                   label)
    ('ce_dice',                    1.0,                     'CE+Dice                     [baseline]'),
    ('plwce_dice',                 best_alpha_with_dice,    f'PLWCE+Dice                  (α={best_alpha_with_dice:.3f}) [baseline]'),
    ('ce_dice_boundary',           1.0,                     'CE+Dice+BL                  [literature]'),
    ('plwce_dice_boundary',        best_alpha_with_dice,    f'PLWCE+Dice+BL               (α={best_alpha_with_dice:.3f})'),
    ('plwce_dice_log_boundary',    best_alpha_with_dice,    f'PLWCE+Dice+LBL              (α={best_alpha_with_dice:.3f})'),
    ('ce_dice_log_boundary',       1.0,                     'CE+Dice+LBL (=Dice+LBL)'),
    ('plwce_boundary',             best_alpha_without_dice, f'PLWCE+BL     (no Dice)       (α={best_alpha_without_dice:.3f})'),
    ('plwce_log_boundary',         best_alpha_without_dice, f'PLWCE+LBL    (no Dice)       (α={best_alpha_without_dice:.3f})'),
]

all_results = {}
for loss_name, alpha, label in experiments:
    model, history, best_dice = train_model(
        loss_name=loss_name, alpha=alpha,
        epochs=FINAL_EPOCHS, lr=FINAL_LR, tag='ba')
    all_results[label] = {
        'model':model, 'history':history, 'best_dice':best_dice,
        'loss_name':loss_name, 'alpha':alpha,
    }

print('\n' + '='*65)
print('[Boundary Ablation 요약 — Val Dice]')
print(f"{'Loss':<52} {'Val Dice':>9}")
print('-'*63)
for label, v in all_results.items():
    print(f"{label:<52} {v['best_dice']:>9.4f}")

In [ ]:
# === Cell 7: 평가 및 결과 저장 ===

COLORS = ['#4878D0','#EE854A','#6ACC65','#D65F5F','#B47CC7','#956CB4','#8C613C','#DC7EC0']

# --- 학습 곡선 ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
for i, (label, v) in enumerate(all_results.items()):
    c = COLORS[i % len(COLORS)]
    ax1.plot(v['history']['loss'],     label=label[:35], color=c)
    ax2.plot(v['history']['val_dice'], label=label[:35], color=c)
ax1.set_title('Training Loss'); ax1.set_xlabel('Epoch'); ax1.legend(fontsize=7)
ax2.set_title('Val Dice');      ax2.set_xlabel('Epoch'); ax2.legend(fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'boundary_ablation_curves.png'), dpi=150)
plt.show()

# --- Test set 정량 평가 ---
print('\n[Test Set 정량 평가]')
print(f"{'Loss':<52} {'Dice':>7} {'Sens':>7} {'Spec':>7} {'AUC':>7}")
print('-' * 80)

final_results = {}
for label, v in all_results.items():
    m = compute_val_metrics(v['model'], test_loader)
    final_results[label] = m
    print(f"{label:<52} {m['Dice']:>7.4f} {m['Sensitivity']:>7.4f} {m['Specificity']:>7.4f} {m['AUC']:>7.4f}")

# --- 바 차트 ---
labels = list(final_results.keys())
dices  = [final_results[l]['Dice'] for l in labels]
idx    = sorted(range(len(dices)), key=lambda i: dices[i], reverse=True)
fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(range(len(labels)), [dices[i] for i in idx],
              color=[COLORS[i%len(COLORS)] for i in range(len(labels))])
ax.set_xticks(range(len(labels)))
ax.set_xticklabels([labels[i][:40] for i in idx], rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Test Dice')
ax.set_title(f'Boundary Ablation — Test Dice ({DOMAIN})')
for bar, val in zip(bars, [dices[i] for i in idx]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
            f'{val:.4f}', ha='center', va='bottom', fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'boundary_ablation_bar.png'), dpi=150)
plt.show()

# --- JSON 저장 ---
with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_boundary_ablation.json'), 'w') as f:
    json.dump({l:{k:v for k,v in m.items()} for l,m in final_results.items()},
              f, indent=2, ensure_ascii=False)
print(f'결과 저장 완료: {RESULTS_DIR}')